# Quick data check

First-pass EDA for a processed dataset: dimensions, column types, missing values, duplicates and simple distributions.


In [ ]:
# Notebook config, environment and logging

import logging
import os
import sys
from pathlib import Path

import pakkenellik.config as conf
from dotenv import load_dotenv
from pakkenellik.log import logger

# Jupyter can run with either the project root or the notebook folder as cwd.
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name in {"templates", "notebooks"}:
    PROJECT_DIR = PROJECT_DIR.parent

project_path = str(PROJECT_DIR)
if project_path not in sys.path:
    sys.path.insert(0, project_path)

ipython = get_ipython()
if ipython and "autoreload" not in ipython.extension_manager.loaded:
    %load_ext autoreload

%autoreload 2

load_dotenv(PROJECT_DIR / ".env")

config = conf.Config(project_path)
logger.setLevel(logging.INFO)

In [ ]:
# Data manipulation
import numpy as np
import pandas as pd

pd.options.display.max_columns = 50
pd.options.display.max_rows = 30

## Load data


In [ ]:
data_path = config.get_processed_file("etl_dataset.csv")
df = pd.read_csv(data_path)
logger.info("Loaded %s rows and %s columns", len(df), len(df.columns))

df.head()

## Structure


In [ ]:
display(df.shape)
display(df.dtypes.to_frame("dtype"))
df.info()

## Missing values and duplicates


In [ ]:
missing = (
    df.isna()
    .sum()
    .to_frame("missing")
    .assign(missing_pct=lambda x: x["missing"] / len(df))
    .sort_values("missing", ascending=False)
)

display(missing)
display({"duplicate_rows": df.duplicated().sum()})

## Numeric summary


In [ ]:
df.describe(include="number").T

## Categorical summary


In [ ]:
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns

for column in categorical_cols[:10]:
    display(df[column].value_counts(dropna=False).head(20).to_frame("count"))

## Date range


In [ ]:
date_columns = [
    column
    for column in df.columns
    if "date" in column.lower() or "dato" in column.lower()
]

for column in date_columns:
    dates = pd.to_datetime(df[column], errors="coerce")
    display(
        pd.DataFrame(
            {
                "column": [column],
                "min": [dates.min()],
                "max": [dates.max()],
                "missing": [dates.isna().sum()],
            }
        )
    )